In [1]:
"Hello"

'Hello'

In [2]:
!uv pip show groq

Name: groq
Version: 1.2.0
Location: c:\Training\26-05-2026\prompt_engineering_evaluation\.venv\Lib\site-packages
Requires: anyio, distro, httpx, pydantic, sniffio, typing-extensions
Required-by:


In [3]:
import sys
print(sys.executable)

c:\Training\26-05-2026\prompt_engineering_evaluation\.venv\Scripts\python.exe


In [1]:
from groq_client import call_llm
from utils import extract_json, validate_output, save_output

import json

llama-3.1-8b-instant


In [2]:
response = call_llm("""
Return ONLY valid JSON.

{
    "message": "hello"
}
""")

print(response)

```json
{
  "message": "hello"
}
```


# **`Zero-Shot Prompting`**

## **Case 1.1 — Zero-Shot Risk Classification for Vendor Onboarding**

In [8]:
from prompts import case_1_1_prompt

In [9]:
case_1_1_response = call_llm(
    prompt=case_1_1_prompt,
    temperature=0.2
)

In [10]:
print(case_1_1_response)

{
    "risk_level": "MEDIUM",
    "key_risk_factors": [
        "Lack of region-specific data residency",
        "Insufficient SOC 2 certification (only Type I)",
        "Unclear rate limits for APIs",
        "Usage-based pricing with potential for significant increases",
        "Limited vendor maturity (only 18 months of operation)",
        "Customer data used for product improvement without explicit opt-in"
    ],
    "missing_information": [
        "Clear documentation of rate limits for APIs",
        "Detailed information on data encryption protocols",
        "Vendor's plan for achieving SOC 2 Type II certification"
    ],
    "business_recommendation": "Proceed with caution, closely monitor vendor performance and pricing, and negotiate a robust SLA with clear rate limits and data protection terms",
    "confidence_score": 0.7
}


In [11]:
case_1_1_parsed = extract_json(
    case_1_1_response
)

case_1_1_parsed

{'risk_level': 'MEDIUM',
 'key_risk_factors': ['Lack of region-specific data residency',
  'Insufficient SOC 2 certification (only Type I)',
  'Unclear rate limits for APIs',
  'Usage-based pricing with potential for significant increases',
  'Limited vendor maturity (only 18 months of operation)',
  'Customer data used for product improvement without explicit opt-in'],
 'missing_information': ['Clear documentation of rate limits for APIs',
  'Detailed information on data encryption protocols',
  "Vendor's plan for achieving SOC 2 Type II certification"],
 'business_recommendation': 'Proceed with caution, closely monitor vendor performance and pricing, and negotiate a robust SLA with clear rate limits and data protection terms',
 'confidence_score': 0.7}

In [12]:
validate_output(case_1_1_parsed)

True

In [13]:
case_1_1_result = {
    "case_name": "case_1_1_zero_shot_vendor_risk",
    "prompt": case_1_1_prompt.strip(),
    "raw_response": case_1_1_response,
    "parsed_output": case_1_1_parsed,
    "valid_json": validate_output(case_1_1_parsed)
}

case_1_1_result

{'case_name': 'case_1_1_zero_shot_vendor_risk',
 'prompt': 'You are an enterprise AI procurement risk analyst.\n\nYour task is to evaluate the following vendor onboarding scenario and classify the overall vendor risk level.\n\nInstructions:\n- Analyze privacy, compliance, operational, pricing, scalability, and vendor maturity risks.\n- Identify implicit risks, not only explicitly stated risks.\n- Avoid generic procurement language.\n- Use only the provided information.\n- Be concise but specific.\n- Return ONLY valid JSON.\n- Do not include markdown formatting.\n- Do not include explanations outside JSON.\n\nRisk Classification Rules:\n- LOW → Minimal operational and compliance concerns.\n- MEDIUM → Noticeable risks requiring monitoring.\n- HIGH → Significant operational, compliance, or business concerns.\n- CRITICAL → Severe risks likely to block onboarding.\n\nRequired JSON Schema:\n{\n    "risk_level": "LOW | MEDIUM | HIGH | CRITICAL",\n    "key_risk_factors": [],\n    "missing_info

In [14]:
save_output(
    "outputs/zero_shot_outputs.json",
    case_1_1_result
)

### Analysis — Case 1.1

#### _Why This Prompt Worked_
- The prompt establishes a specialized enterprise procurement analyst role.
- Clear risk dimensions guide the model toward structured reasoning.
- The schema constraint improves output consistency and JSON reliability.
- The prompt encourages implicit risk identification instead of shallow summarization.

#### _Potential Failure Modes_
- The model may inconsistently weigh privacy versus operational risks.
- Confidence score calibration may vary between runs.
- Some responses may still contain markdown formatting despite instructions.

#### _Possible Improvements_
- Add numerical scoring criteria for each risk category.
- Add stricter confidence score guidance.
- Add schema-level validation using Pydantic.

## **Case 1.2 — Zero-Shot Executive Decision Memo**

In [15]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [16]:
from prompts import case_1_2_prompt

In [17]:
case_1_2_response = call_llm(
    prompt=case_1_2_prompt,
    temperature=0.2
)

In [18]:
case_1_2_parsed = extract_json(
    case_1_2_response
)

case_1_2_parsed

{'decision': 'APPROVE_WITH_CONDITIONS',
 'rationale': 'The proposed GenAI chatbot can help reduce the ticket load and improve response times, but concerns around compliance, job losses, and lack of AI governance policies need to be addressed.',
 'financial_considerations': ['Estimated implementation cost of $250,000',
  'Ongoing monthly cost of $30,000',
  'Potential ROI through reduced ticket load and improved response times',
  'Payback period of 12 months as required by the CFO'],
 'operational_considerations': ['Reduction in ticket load by 35% as estimated by the CTO',
  'Improved response times from 11 hours to a target of 3 hours',
  'Impact on support team workflows and processes'],
 'people_impact': ['Potential job losses among the 120 human agents',
  'Need for retraining or upskilling of agents to work with the chatbot'],
 'compliance_risks': ['Handling of personal information in customer support tickets',
  'Need for data protection and privacy measures'],
 'conditions_for_a

In [19]:
validate_output(case_1_2_parsed)

True

In [20]:
case_1_2_result = {
    "case_name": "case_1_2_zero_shot_executive_memo",
    "prompt": case_1_2_prompt,
    "raw_response": case_1_2_response,
    "parsed_output": case_1_2_parsed,
    "valid_json": validate_output(case_1_2_parsed)
}

save_output(
    "outputs/zero_shot_outputs.json",
    case_1_2_result
)

### Analysis — Case 1.2

#### _Why This Prompt Worked_
- The prompt frames the model as a senior enterprise strategy advisor, encouraging executive-level reasoning instead of simple summarization.
- Explicit decision rules guide the model toward a clear business recommendation.
- The instructions force consideration of governance, ROI, compliance, operational scalability, and workforce impact together.
- The schema constraint improves consistency and makes the output easier to validate and review.
- The prompt discourages unrealistic automation claims and encourages balanced analysis.

#### _Potential Failure Modes_
- The model may overestimate projected chatbot efficiency without sufficient evidence.
- Financial reasoning may remain qualitative because ROI calculation details are not explicitly requested.
- The model may underemphasize governance gaps despite missing AI governance policies.
- Some responses may become overly cautious due to compliance concerns involving personal data.

#### _Possible Improvements_
- Add explicit ROI estimation instructions using implementation and operating costs.
- Add risk weighting for governance and compliance readiness.
- Introduce stricter conditions for approval thresholds.
- Add schema-level validation for decision enums and confidence consistency.

# **`Few-Shot Prompting`**

In [21]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [22]:
from prompts import case_2_1_prompt

In [23]:
case_2_1_response = call_llm(
    prompt=case_2_1_prompt,
    temperature=0.2
)

In [24]:
case_2_1_parsed = extract_json(
    case_2_1_response
)

case_2_1_parsed

[{'ticket': 'I was charged twice this month and your support team has not replied for five days. I am going to post this on social media if this is not fixed today.',
  'category': 'BILLING_ISSUE',
  'priority': 'URGENT',
  'justification': 'The primary issue is duplicate billing with significantly delayed support response and potential public escalation.'},
 {'ticket': 'The export button stopped working after your latest update. Our reporting team is blocked.',
  'category': 'TECHNICAL_BUG',
  'priority': 'HIGH',
  'justification': 'The issue describes broken application functionality after an update affecting the reporting team.'},
 {'ticket': 'Can you add approval workflows before invoices are submitted?',
  'category': 'FEATURE_REQUEST',
  'priority': 'MEDIUM',
  'justification': 'The customer is requesting a new workflow capability for invoice submission.'},
 {'ticket': 'We need confirmation that our customer data is not being used to train your AI models.',
  'category': 'COMPLIA

In [25]:
validate_output(case_2_1_parsed)

True

In [26]:
case_2_1_result = {
    "case_name": "case_2_1_few_shot_ticket_classification",
    "prompt": case_2_1_prompt,
    "raw_response": case_2_1_response,
    "parsed_output": case_2_1_parsed,
    "valid_json": validate_output(case_2_1_parsed)
}

save_output(
    "outputs/few_shot_outputs.json",
    case_2_1_result
)

### Analysis — Case 2.1

#### _Why This Prompt Worked_
- The prompt uses multiple labeled examples to teach category boundaries instead of relying only on direct definitions.
- Examples were intentionally selected to cover ambiguity between billing issues, escalation behavior, compliance concerns, and technical failures.
- The instructions explicitly separate emotional tone from actual ticket intent, helping the model avoid incorrectly classifying angry tickets as escalation risks.
- The examples reinforce that priority and category are different dimensions.
- The structured JSON schema improves output consistency and simplifies parsing.

#### _Potential Failure Modes_
- The model may still incorrectly classify emotionally aggressive tickets as ESCALATION_RISK instead of their primary operational category.
- ACCOUNT_ACCESS and TECHNICAL_BUG may overlap in authentication-related failures.
- Some responses may generate inconsistent priority levels depending on perceived business impact.
- The model may occasionally classify AI data usage concerns as TECHNICAL_BUG instead of COMPLIANCE_CONCERN.

#### _Possible Improvements_
- Add more edge-case examples involving mixed billing and escalation language.
- Include examples where multiple categories appear possible but only one is correct.
- Add stricter priority scoring rules based on operational severity and business impact.
- Add schema-level validation for category enums and priority consistency.

## Case 2.2 — Few-Shot Transformation from Requirement to API Contract

In [27]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [28]:
from prompts import case_2_2_prompt

In [29]:
case_2_2_response = call_llm(
    prompt=case_2_2_prompt,
    temperature=0.2
)
case_2_2_response

'[\n    {\n        "action": "APPLY_LEAVE",\n        "parameters": {\n            "leave_type": "",\n            "start_date": "12th June",\n            "end_date": "15th June",\n            "reason": "travelling"\n        },\n        "requires_clarification": true,\n        "clarification_question": "Please specify the leave type.",\n        "confidence": 0.9\n    },\n    {\n        "action": "CHECK_BALANCE",\n        "parameters": {\n            "leave_type": "casual"\n        },\n        "requires_clarification": false,\n        "clarification_question": "",\n        "confidence": 0.98\n    },\n    {\n        "action": "CANCEL_LEAVE",\n        "parameters": {},\n        "requires_clarification": true,\n        "clarification_question": "Please specify the exact date for next Friday.",\n        "confidence": 0.82\n    },\n    {\n        "action": "GET_POLICY",\n        "parameters": {\n            "leave_type": "maternity"\n        },\n        "requires_clarification": false,\n      

In [30]:
case_2_2_parsed = extract_json(
    case_2_2_response
)

case_2_2_parsed

[{'action': 'APPLY_LEAVE',
  'parameters': {'leave_type': '',
   'start_date': '12th June',
   'end_date': '15th June',
   'reason': 'travelling'},
  'requires_clarification': True,
  'clarification_question': 'Please specify the leave type.',
  'confidence': 0.9},
 {'action': 'CHECK_BALANCE',
  'parameters': {'leave_type': 'casual'},
  'requires_clarification': False,
  'clarification_question': '',
  'confidence': 0.98},
 {'action': 'CANCEL_LEAVE',
  'parameters': {},
  'requires_clarification': True,
  'clarification_question': 'Please specify the exact date for next Friday.',
  'confidence': 0.82},
 {'action': 'GET_POLICY',
  'parameters': {'leave_type': 'maternity'},
  'requires_clarification': False,
  'clarification_question': '',
  'confidence': 0.97},
 {'action': 'APPLY_LEAVE',
  'parameters': {},
  'requires_clarification': True,
  'clarification_question': 'Please specify the exact leave dates and leave type.',
  'confidence': 0.75}]

In [31]:
print(case_2_2_parsed)

[{'action': 'APPLY_LEAVE', 'parameters': {'leave_type': '', 'start_date': '12th June', 'end_date': '15th June', 'reason': 'travelling'}, 'requires_clarification': True, 'clarification_question': 'Please specify the leave type.', 'confidence': 0.9}, {'action': 'CHECK_BALANCE', 'parameters': {'leave_type': 'casual'}, 'requires_clarification': False, 'clarification_question': '', 'confidence': 0.98}, {'action': 'CANCEL_LEAVE', 'parameters': {}, 'requires_clarification': True, 'clarification_question': 'Please specify the exact date for next Friday.', 'confidence': 0.82}, {'action': 'GET_POLICY', 'parameters': {'leave_type': 'maternity'}, 'requires_clarification': False, 'clarification_question': '', 'confidence': 0.97}, {'action': 'APPLY_LEAVE', 'parameters': {}, 'requires_clarification': True, 'clarification_question': 'Please specify the exact leave dates and leave type.', 'confidence': 0.75}]


In [32]:
case_2_2_result = {
    "case_name": "case_2_2_few_shot_api_contract",
    "prompt": case_2_2_prompt,
    "raw_response": case_2_2_response,
    "parsed_output": case_2_2_parsed,
    "valid_json": validate_output(case_2_2_parsed)
}

case_2_2_result

{'case_name': 'case_2_2_few_shot_api_contract',
 'prompt': 'You are an AI leave-management request parser.\n\nYour task is to convert natural language leave-management requests into structured API contracts.\n\nSupported Actions:\n- APPLY_LEAVE\n- CHECK_BALANCE\n- CANCEL_LEAVE\n- GET_POLICY\n\nInstructions:\n- Do not invent dates, leave types, or reasons.\n- If required information is missing or ambiguous, set requires_clarification to true.\n- Ambiguous phrases like "next Friday" or "sometime next week" require clarification.\n- Return ONLY valid JSON.\n- Do not include markdown formatting.\n- Do not include explanations outside JSON.\n\nRequired Output Schema:\n[\n    {\n        "action": "",\n        "parameters": {},\n        "requires_clarification": true,\n        "clarification_question": "",\n        "confidence": 0.0\n    }\n]\n\nExamples:\n\nUser Request:\n"I want to take casual leave from 12 June to 15 June because I am travelling."\n\nOutput:\n{\n    "action": "APPLY_LEAVE"

In [33]:
save_output(
    "outputs/few_shot_outputs.json",
    case_2_2_result
)

### Analysis — Case 2.2

#### _Why This Prompt Worked_
- The prompt uses few-shot examples to teach both successful API extraction behavior and clarification behavior.
- The examples intentionally include incomplete and ambiguous requests to train the model not to hallucinate missing parameters.
- Relative date phrases such as "next Friday" and "sometime next week" reinforce ambiguity handling rules.
- The instructions explicitly prohibit inventing dates or leave details, improving output reliability.
- The JSON array schema aligns correctly with multiple user requests and improves parsing consistency.

#### _Potential Failure Modes_
- The model may still attempt to infer dates for ambiguous temporal expressions.
- Some responses may partially populate parameters even when clarification is required.
- Confidence scores may vary inconsistently across similar ambiguity levels.
- APPLY_LEAVE and CANCEL_LEAVE requests may overlap when leave periods are unclear.

#### _Possible Improvements_
- Add more edge-case examples involving partial leave details and overlapping requests.
- Introduce stricter rules for interpreting relative dates.
- Add action-specific parameter validation requirements.
- Add schema-level validation using Pydantic models for stronger API contract enforcement.

# **`Chain-of-Thought Prompting`**

## **Case 3.1 — Business ROI Decision with Hidden Trade-Offs**

In [34]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [35]:
from prompts import case_3_1_prompt

In [36]:
case_3_1_response = call_llm(
    prompt=case_3_1_prompt,
    temperature=0.2
)

In [37]:
case_3_1_parsed = extract_json(
    case_3_1_response
)

case_3_1_parsed

{'incremental_revenue_range': '$80,000 - $140,000',
 'incremental_gross_profit_range': '$32,000 - $56,000',
 'monthly_net_benefit_range': '$2,000 - $26,000',
 'payback_period_range_months': '7 - 9 months',
 'decision': 'APPROVE',
 'reasoning_summary': 'The expected payback period is within the required 12 months, and the monthly net benefit is positive, indicating the project is likely to generate sufficient returns to justify the investment.',
 'key_assumptions': ['Revenue uplift ranges from 4% to 7%',
  'Gross margin remains constant at 40%',
  'Implementation time is 3 months',
  'Ongoing costs include $22,000 for AI infrastructure and $8,000 for maintenance']}

In [38]:
validate_output(case_3_1_parsed)

True

In [39]:
case_3_1_result = {
    "case_name": "case_3_1_cot_roi_decision",
    "prompt": case_3_1_prompt,
    "raw_response": case_3_1_response,
    "parsed_output": case_3_1_parsed,
    "valid_json": validate_output(case_3_1_parsed)
}

In [40]:
case_3_1_result

{'case_name': 'case_3_1_cot_roi_decision',
 'prompt': 'You are a senior AI business strategy analyst.\n\nYour task is to evaluate whether an AI recommendation engine project should be approved.\n\nInstructions:\n- Perform careful numerical reasoning before making the decision.\n- Use gross profit, NOT total revenue, when evaluating payback.\n- Subtract ongoing AI operating costs from projected gains.\n- Treat implementation time separately from post-launch payback.\n- Think step-by-step internally, but return only concise reasoning summaries.\n- Use only the provided information.\n- Return ONLY valid JSON.\n- Do not include markdown formatting.\n- Do not include explanations outside JSON.\n\nRequired JSON Schema:\n{\n    "incremental_revenue_range": "",\n    "incremental_gross_profit_range": "",\n    "monthly_net_benefit_range": "",\n    "payback_period_range_months": "",\n    "decision": "APPROVE | REJECT | APPROVE_WITH_CONDITIONS",\n    "reasoning_summary": "",\n    "key_assumptions"

In [41]:
save_output(
    "outputs/cot_outputs.json",
    case_3_1_result
)

### Analysis — Case 3.1

#### _Why This Prompt Worked_
- The prompt explicitly instructs the model to use gross profit rather than total revenue for payback analysis.
- Operational AI costs are separated from implementation cost, improving financial realism.
- The reasoning-oriented instructions encourage structured numerical evaluation before decision making.
- The schema structure forces the model to expose intermediate business calculations rather than giving only a final recommendation.
- The prompt discourages shallow ROI reasoning and reinforces implementation timeline considerations.

#### _Potential Failure Modes_
- The model may still incorrectly calculate payback using revenue instead of gross profit.
- Monthly operating costs may be omitted from net benefit calculations.
- Some responses may oversimplify uncertainty in projected revenue uplift.
- The model may inconsistently interpret whether payback includes implementation delay.

#### _Possible Improvements_
- Add explicit formulas for payback calculation.
- Add stricter numerical formatting requirements.
- Include sensitivity analysis for low and high adoption scenarios.
- Add validation rules for financial consistency between calculated fields.

## **Case 3.2 — Root Cause Analysis for ML Model Performance Drop**

In [42]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [43]:
from prompts import case_3_2_prompt

In [44]:
case_3_2_response = call_llm(
    prompt=case_3_2_prompt,
    temperature=0.2
)

In [45]:
case_3_2_parsed = extract_json(
    case_3_2_response
)

case_3_2_parsed

{'most_likely_causes': ['data drift', 'concept drift'],
 'evidence': ['Transaction volume increased by 30%',
  'New payment channel was introduced',
  'Fraud patterns changed after a promotional campaign',
  'Feature distribution for transaction_amount shifted significantly'],
 'less_likely_causes': ['threshold miscalibration'],
 'recommended_diagnostics': ['Monitor feature distributions',
  'Analyze fraud patterns',
  'Evaluate model performance on new payment channel',
  'Compare model predictions with actual outcomes'],
 'short_term_actions': ['Adjust model thresholds',
  'Implement data quality checks'],
 'long_term_actions': ['Retrain model with new data',
  'Update model to incorporate new payment channel and fraud patterns'],
 'reasoning_summary': 'The significant changes in transaction volume, introduction of a new payment channel, and shifts in fraud patterns and feature distributions suggest data drift and concept drift as the primary causes of model performance degradation.'

In [46]:
validate_output(case_3_2_parsed)

True

In [47]:
case_3_2_result = {
    "case_name": "case_3_2_cot_ml_root_cause_analysis",
    "prompt": case_3_2_prompt,
    "raw_response": case_3_2_response,
    "parsed_output": case_3_2_parsed,
    "valid_json": validate_output(case_3_2_parsed)
}

In [48]:
case_3_2_result

{'case_name': 'case_3_2_cot_ml_root_cause_analysis',
 'prompt': 'You are a senior machine learning systems analyst.\n\nYour task is to perform a structured root cause analysis for an ML fraud detection model whose performance degraded after deployment.\n\nInstructions:\n- Carefully distinguish between:\n  - data drift\n  - concept drift\n  - pipeline failure\n  - threshold miscalibration\n- Do not assume pipeline failure unless evidence supports it.\n- Recommend concrete diagnostics and remediation steps.\n- Avoid generic "just retrain the model" responses.\n- Think step-by-step internally, but return only concise reasoning summaries.\n- Use only the provided information.\n- Return ONLY valid JSON.\n- Do not include markdown formatting.\n- Do not include explanations outside JSON.\n\nRequired JSON Schema:\n{\n    "most_likely_causes": [],\n    "evidence": [],\n    "less_likely_causes": [],\n    "recommended_diagnostics": [],\n    "short_term_actions": [],\n    "long_term_actions": [],\

In [49]:
save_output(
    "outputs/cot_outputs.json",
    case_3_2_result
)

### Analysis — Case 3.2

#### _Why This Prompt Worked_
- The prompt explicitly separates different ML failure modes, guiding the model toward structured diagnostic reasoning.
- Instructions discourage unsupported assumptions about pipeline failures despite degraded performance.
- The scenario provides multiple correlated signals such as feature distribution shifts, new payment channels, and evolving fraud patterns, encouraging deeper reasoning.
- The reasoning-oriented structure improves causal analysis instead of simplistic retraining recommendations.
- The schema encourages separation of immediate remediation and long-term operational improvements.

#### _Potential Failure Modes_
- The model may over-prioritize retraining without sufficient diagnostic validation.
- Data drift and concept drift may sometimes be conflated.
- Threshold miscalibration may be ignored despite precision decline.
- Some responses may incorrectly interpret increased transaction volume as direct evidence of pipeline instability.

#### _Possible Improvements_
- Add explicit requirements for drift monitoring metrics.
- Include confusion matrix changes for deeper reasoning.
- Add stricter differentiation criteria between feature drift and behavioral fraud drift.
- Introduce quantitative monitoring thresholds for retraining triggers.

# **`LLM-as-Judge`**

## **Case 4.1 — Judging AI-Generated Customer Support Responses**

In [50]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [51]:
from prompts import case_4_1_prompt

In [52]:
case_4_1_response = call_llm(
    prompt=case_4_1_prompt,
    temperature=0.2
)

In [53]:
case_4_1_parsed = extract_json(
    case_4_1_response
)

case_4_1_parsed

{'response_a': {'scores': {'Empathy': 2,
   'Professionalism': 4,
   'Problem Resolution Quality': 2,
   'Clarity': 3,
   'Policy Safety': 5},
  'strengths': [],
  'weaknesses': ['Lack of empathy', 'Vague solution']},
 'response_b': {'scores': {'Empathy': 5,
   'Professionalism': 5,
   'Problem Resolution Quality': 5,
   'Clarity': 5,
   'Policy Safety': 5},
  'strengths': ['Empathetic tone',
   'Clear guidance',
   'Proper escalation handling'],
  'weaknesses': []},
 'winner': 'B',
 'judge_reasoning_summary': 'Response B demonstrates a higher level of empathy, provides clear and actionable guidance, and handles the escalation properly, making it a more effective and customer-centric response.'}

In [54]:
validate_output(case_4_1_parsed)

True

In [55]:
case_4_1_result = {
    "case_name": "case_4_1_llm_judge_customer_support",
    "prompt": case_4_1_prompt,
    "raw_response": case_4_1_response,
    "parsed_output": case_4_1_parsed,
    "valid_json": validate_output(case_4_1_parsed)
}

In [56]:
case_4_1_result

{'case_name': 'case_4_1_llm_judge_customer_support',
 'prompt': 'You are an expert customer support quality evaluator.\n\nYour task is to evaluate two customer support responses using a structured scoring rubric.\n\nEvaluation Criteria:\n- Empathy\n- Professionalism\n- Problem Resolution Quality\n- Clarity\n- Policy Safety\n\nScoring Rules:\n- Score each category from 1 to 5.\n- Do not prefer a response simply because it is longer.\n- Penalize vague or dismissive responses.\n- Penalize unsafe promises such as guaranteeing refunds without verification.\n- Reward empathy, actionable guidance, and escalation handling.\n\nReturn ONLY valid JSON.\nDo not include markdown formatting.\nDo not include explanations outside JSON.\n\nRequired JSON Schema:\n{\n    "response_a": {\n        "scores": {},\n        "strengths": [],\n        "weaknesses": []\n    },\n    "response_b": {\n        "scores": {},\n        "strengths": [],\n        "weaknesses": []\n    },\n    "winner": "A | B | TIE",\n   

In [57]:
save_output(
    "outputs/llm_judge_outputs.json",
    case_4_1_result
)

### Analysis — Case 4.1

#### _Why This Prompt Worked_
- The prompt defines a clear evaluation rubric, improving scoring consistency.
- Explicit scoring criteria reduce subjective preference and encourage structured comparison.
- Instructions discourage length bias and unsafe refund promises, reinforcing realistic customer support evaluation.
- The schema structure encourages separation of strengths, weaknesses, and quantitative scoring.
- The evaluator role shifts the model from generation behavior into analytical judgment behavior.

#### _Potential Failure Modes_
- The model may still implicitly favor longer responses despite instructions.
- Scoring consistency may vary across runs for subjective categories such as empathy.
- Some evaluations may over-penalize cautious policy language.
- Weakness explanations may become repetitive across responses.

#### _Possible Improvements_
- Add weighted scoring for critical categories such as resolution quality and policy safety.
- Define stricter scoring anchors for each numeric level.
- Add calibration examples showing acceptable versus unacceptable responses.
- Introduce tie-breaking rules for closely scored responses.

## **Case 4.2 — Judging Code Explanation Quality**

In [58]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [59]:
from prompts import case_4_2_prompt

In [60]:
case_4_2_response = call_llm(
    prompt=case_4_2_prompt,
    temperature=0.2
)

In [61]:
case_4_2_parsed = extract_json(
    case_4_2_response
)

case_4_2_parsed

{'explanation_a': {'scores': {'Technical Accuracy': 5,
   'Beginner Friendliness': 4,
   'Clarity': 5,
   'Completeness': 4,
   'Misleading Statements': 1},
  'issues': [],
  'overall_score': 4.6},
 'explanation_b': {'scores': {'Technical Accuracy': 2,
   'Beginner Friendliness': 3,
   'Clarity': 3,
   'Completeness': 2,
   'Misleading Statements': 5},
  'issues': ['oversimplification', 'incorrect generalization'],
  'overall_score': 2.2},
 'winner': 'explanation_a',
 'judge_reasoning_summary': 'Explanation A is more accurate and clear, while Explanation B contains misleading statements and oversimplifications. Deep copy is not always better because it can be slower and more memory-intensive, and shallow copy can be sufficient when nested objects do not need to be modified.'}

In [62]:
case_4_2_result = {
    "case_name": "case_4_2_llm_judge_code_explanation",
    "prompt": case_4_2_prompt,
    "raw_response": case_4_2_response,
    "parsed_output": case_4_2_parsed,
    "valid_json": validate_output(case_4_2_parsed)
}

In [63]:
case_4_2_result

{'case_name': 'case_4_2_llm_judge_code_explanation',
 'prompt': 'You are an expert programming educator and technical reviewer.\n\nYour task is to evaluate two explanations about shallow copy and deep copy in Python.\n\nEvaluation Criteria:\n- Technical Accuracy\n- Beginner Friendliness\n- Clarity\n- Completeness\n- Misleading Statements\n\nScoring Rules:\n- Score each category from 1 to 5.\n- Penalize technically incorrect simplifications.\n- Do not reward oversimplification if it reduces correctness.\n- Explain why deep copy is not always better.\n- Focus on educational usefulness for beginner developers.\n\nReturn ONLY valid JSON.\nDo not include markdown formatting.\nDo not include explanations outside JSON.\n\nRequired JSON Schema:\n{\n    "explanation_a": {\n        "scores": {},\n        "issues": [],\n        "overall_score": 0\n    },\n    "explanation_b": {\n        "scores": {},\n        "issues": [],\n        "overall_score": 0\n    },\n    "winner": "",\n    "judge_reasoni

In [64]:
save_output(
    "outputs/llm_judge_outputs.json",
    case_4_2_result
)

### Analysis — Case 4.2

#### _Why This Prompt Worked_
- The prompt establishes a technical educator evaluation role, encouraging educational reasoning instead of subjective preference.
- The scoring rubric separates clarity from technical correctness, helping prevent inaccurate simplifications from receiving high scores.
- Explicit instructions discourage rewarding oversimplified but misleading explanations.
- The schema structure encourages structured critique instead of generic preference statements.
- The prompt reinforces beginner usefulness while preserving technical accuracy.

#### _Potential Failure Modes_
- The model may still overvalue simplicity despite technical inaccuracies.
- Some evaluations may insufficiently explain why deep copy is not universally preferable.
- Scoring differences between clarity and correctness may become inconsistent across runs.
- The evaluator may under-discuss performance and memory trade-offs of deep copying.

#### _Possible Improvements_
- Add explicit scoring anchors for technical accuracy severity.
- Include examples of acceptable beginner simplifications versus misleading oversimplifications.
- Add evaluation dimensions for performance awareness and practical usage guidance.
- Introduce weighted scoring for correctness over presentation quality.

# **`Self-Consistency Prompting`**

## **Case 5.1 — Self-Consistency for Complex Policy Interpretation**

In [65]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [66]:
from prompts import case_5_1_prompt

In [67]:
case_5_1_runs = []

for i in range(5):

    response = call_llm(
        prompt=case_5_1_prompt,
        temperature=0.8
    )

    parsed = extract_json(response)

    case_5_1_runs.append({
        "run": i + 1,
        "raw_response": response,
        "parsed_output": parsed
    })

In [68]:
case_5_1_runs

[{'run': 1,
  'raw_response': '{\n    "reimbursable_amount": 45.00,\n    "reasoning_summary": "International travel with same-day return, daily meal limit increased by 25% to $75, 50% of $75 is $37.50, alcohol excluded, claim limited to $37.50, but employee claimed $70 - $12 (alcohol) = $58, which exceeds 50% limit, so reimbursed at $37.50 * 0.75 (75% of daily meal limit for international) is not applied, instead 50% of increased limit is the basis, thus 50% of $75 is the correct calculation"\n}',
  'parsed_output': {'reimbursable_amount': 45.0,
   'reasoning_summary': 'International travel with same-day return, daily meal limit increased by 25% to $75, 50% of $75 is $37.50, alcohol excluded, claim limited to $37.50, but employee claimed $70 - $12 (alcohol) = $58, which exceeds 50% limit, so reimbursed at $37.50 * 0.75 (75% of daily meal limit for international) is not applied, instead 50% of increased limit is the basis, thus 50% of $75 is the correct calculation'}},
 {'run': 2,
  'ra

In [69]:
amounts = []

for run in case_5_1_runs:

    parsed = run["parsed_output"]

    if parsed and "reimbursable_amount" in parsed:

        amounts.append(
            parsed["reimbursable_amount"]
        )

amounts

[45.0, 45.0, 45.0, 45, 45]

In [70]:
from collections import Counter

vote_counts = Counter(amounts)

vote_counts

Counter({45.0: 5})

In [71]:
final_amount = vote_counts.most_common(1)[0][0]

final_amount

45.0

In [72]:
case_5_1_result = {
    "case_name": "case_5_1_self_consistency_policy",
    "individual_answers": case_5_1_runs,
    "final_reimbursable_amount": final_amount,
    "consistency_count": dict(vote_counts),
    "final_decision": f"Most consistent reimbursable amount is {final_amount}",
    "reasoning_summary": "Majority voting was used across five independent reasoning runs."
}

In [73]:
save_output(
    "outputs/self_consistency_outputs.json",
    case_5_1_result
)

### Analysis — Case 5.1

#### _Why This Prompt Worked_
- The prompt explicitly reinforces careful policy interpretation and multi-rule reasoning.
- Independent high-temperature runs encourage reasoning diversity for self-consistency aggregation.
- Majority voting reduces the impact of isolated reasoning mistakes.
- The structured schema simplifies extraction of reimbursable amounts across runs.
- The prompt reinforces exclusion of alcohol expenses and conditional reimbursement logic.

#### _Potential Failure Modes_
- The model may inconsistently apply international uplift before or after same-day adjustment.
- Some runs may incorrectly reimburse alcohol expenses.
- Numerical outputs may vary slightly due to reasoning path differences.
- The model may inconsistently interpret reimbursement caps.

#### _Possible Improvements_
- Add explicit rule precedence ordering.
- Add numerical calculation formatting requirements.
- Increase run count for stronger consistency confidence.
- Add automatic disagreement analysis for divergent outputs.

## **Case 5.2 — Self-Consistency for Logical Deduction**

In [74]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [75]:
from prompts import case_5_2_prompt

In [76]:
case_5_2_runs = []

for i in range(5):

    response = call_llm(
        prompt=case_5_2_prompt,
        temperature=0.8
    )

    parsed = extract_json(response)

    case_5_2_runs.append({
        "run": i + 1,
        "raw_response": response,
        "parsed_output": parsed
    })

In [77]:
case_5_2_runs

[{'run': 1,
  'raw_response': '{\n    "risk_level": "MEDIUM",\n    "reasoning_summary": "User logged in outside business hours and failed MFA once, country is a known VPN country so not considered new"\n}',
  'parsed_output': {'risk_level': 'MEDIUM',
   'reasoning_summary': 'User logged in outside business hours and failed MFA once, country is a known VPN country so not considered new'}},
 {'run': 2,
  'raw_response': '{\n    "risk_level": "MEDIUM",\n    "reasoning_summary": "User logged in outside business hours and failed MFA once, but Germany is a known VPN country so it\'s not considered a new country for HIGH risk conditions"\n}',
  'parsed_output': {'risk_level': 'MEDIUM',
   'reasoning_summary': "User logged in outside business hours and failed MFA once, but Germany is a known VPN country so it's not considered a new country for HIGH risk conditions"}},
 {'run': 3,
  'raw_response': '{\n    "risk_level": "MEDIUM",\n    "reasoning_summary": "User logged in outside business hours 

In [78]:
risk_levels = []

for run in case_5_2_runs:

    parsed = run["parsed_output"]

    if parsed and "risk_level" in parsed:

        risk_levels.append(
            parsed["risk_level"]
        )

risk_levels

['MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM']

In [79]:
from collections import Counter

risk_votes = Counter(risk_levels)

risk_votes

Counter({'MEDIUM': 5})

In [80]:
final_risk_level = risk_votes.most_common(1)[0][0]

final_risk_level

'MEDIUM'

In [81]:
disagreement_analysis = ""

if len(risk_votes) > 1:

    disagreement_analysis = (
        "Some runs incorrectly classified HIGH risk by treating Germany "
        "as a new country despite existing known-country and VPN rules."
    )

else:

    disagreement_analysis = (
        "All runs consistently applied the access rules."
    )

In [82]:
case_5_2_result = {
    "case_name": "case_5_2_self_consistency_security_risk",
    "runs": case_5_2_runs,
    "risk_level_votes": dict(risk_votes),
    "final_risk_level": final_risk_level,
    "disagreement_analysis": disagreement_analysis,
    "final_reasoning_summary": "Majority voting was used across five independent reasoning runs."
}

In [83]:
save_output(
    "outputs/self_consistency_outputs.json",
    case_5_2_result
)

### Analysis — Case 5.2

#### _Why This Prompt Worked_
- The prompt reinforces strict rule-based reasoning instead of intuitive threat assumptions.
- Independent high-temperature runs help expose reasoning inconsistencies for aggregation analysis.
- The instructions explicitly discourage assuming HIGH risk without satisfying all required conditions.
- The schema structure simplifies extraction and majority voting of final classifications.
- The disagreement analysis helps identify where reasoning errors occurred across runs.

#### _Potential Failure Modes_
- Some runs may incorrectly classify HIGH risk due to the large file download count.
- The model may ignore the known-country or VPN-country conditions.
- CRITICAL escalation logic may occasionally be triggered incorrectly.
- Different reasoning paths may produce inconsistent interpretations of VPN rules.

#### _Possible Improvements_
- Add explicit intermediate condition evaluation fields.
- Add rule-by-rule boolean outputs before final classification.
- Increase run count for stronger aggregation confidence.
- Add automatic reasoning comparison between inconsistent runs.

# **`Tree-of-Thought Prompting`**

## **Case 6.1 — Selecting the Best AI Automation Use Case**

In [84]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [85]:
from prompts import case_6_1_prompt

In [86]:
case_6_1_response = call_llm(
    prompt=case_6_1_prompt,
    temperature=0.3
)

In [87]:
print(case_6_1_response)

{
    "options_evaluated": [
        {
            "option": "AI customer support assistant",
            "business_value_score": 4,
            "feasibility_score": 3,
            "risk_score": 2,
            "pilot_suitability_score": 4,
            "adoption_score": 3,
            "overall_score": 3.2,
            "trade_offs": ["high ticket volume", "moderate implementation complexity", "personal customer data"]
        },
        {
            "option": "AI sales proposal generator",
            "business_value_score": 4,
            "feasibility_score": 5,
            "risk_score": 5,
            "pilot_suitability_score": 5,
            "adoption_score": 5,
            "overall_score": 4.8,
            "trade_offs": ["medium usage frequency", "low data sensitivity"]
        },
        {
            "option": "AI contract risk analyzer",
            "business_value_score": 5,
            "feasibility_score": 2,
            "risk_score": 1,
            "pilot_suitability_score": 2

In [88]:
case_6_1_parsed = extract_json(
    case_6_1_response
)

case_6_1_parsed

{'options_evaluated': [{'option': 'AI customer support assistant',
   'business_value_score': 4,
   'feasibility_score': 3,
   'risk_score': 2,
   'pilot_suitability_score': 4,
   'adoption_score': 3,
   'overall_score': 3.2,
   'trade_offs': ['high ticket volume',
    'moderate implementation complexity',
    'personal customer data']},
  {'option': 'AI sales proposal generator',
   'business_value_score': 4,
   'feasibility_score': 5,
   'risk_score': 5,
   'pilot_suitability_score': 5,
   'adoption_score': 5,
   'overall_score': 4.8,
   'trade_offs': ['medium usage frequency', 'low data sensitivity']},
  {'option': 'AI contract risk analyzer',
   'business_value_score': 5,
   'feasibility_score': 2,
   'risk_score': 1,
   'pilot_suitability_score': 2,
   'adoption_score': 3,
   'overall_score': 2.6,
   'trade_offs': ['high business value',
    'high legal sensitivity',
    'high implementation complexity']},
  {'option': 'AI internal HR policy assistant',
   'business_value_score': 

In [89]:
validate_output(case_6_1_parsed)

True

In [90]:
case_6_1_result = {
    "case_name": "case_6_1_tree_of_thought_ai_use_case",
    "prompt": case_6_1_prompt,
    "raw_response": case_6_1_response,
    "parsed_output": case_6_1_parsed,
    "valid_json": validate_output(case_6_1_parsed)
}

save_output(
    "outputs/tree_of_thought_outputs.json",
    case_6_1_result
)

### Analysis — Case 6.1

#### _Why This Prompt Worked_
- The prompt explicitly instructs the model to evaluate each option independently before synthesis, reinforcing tree-of-thought reasoning behavior.
- Multiple evaluation dimensions prevent simplistic decision making based only on business value.
- The scoring structure encourages structured comparison between feasibility, risk, adoption, and pilot suitability.
- The prompt reinforces trade-off analysis rather than isolated scoring.
- The schema structure makes branch-wise evaluation easier to review and compare.

#### _Potential Failure Modes_
- The model may still overweight business value relative to implementation complexity.
- Some scores may become inconsistent across dimensions due to subjective interpretation.
- Risk scoring may occasionally invert if the model forgets that lower operational risk should receive higher scores.
- Trade-off explanations may become repetitive across options.

#### _Possible Improvements_
- Add weighted scoring formulas for overall evaluation.
- Introduce stricter definitions for each scoring dimension.
- Add calibration examples for high versus low pilot suitability.
- Add sensitivity analysis for implementation timelines and compliance exposure.

# **Case 6.2 — Tree-of-Thought Architecture Selection**

In [91]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [92]:
from prompts import case_6_2_prompt

In [93]:
case_6_2_response = call_llm(
    prompt=case_6_2_prompt,
    temperature=0.3
)

In [94]:
print(case_6_2_response)

{
    "architecture_scores": [
        {"option": "A", "score": 8, "rationale": "Balances accuracy and complexity"},
        {"option": "B", "score": 6, "rationale": "Fine-tuning may not be feasible with frequent document changes"},
        {"option": "C", "score": 4, "rationale": "Lacks accuracy due to no LLM"},
        {"option": "D", "score": 9, "rationale": "High accuracy and citation reliability"}
    ],
    "recommended_architecture": "Option D",
    "implementation_rationale": "Agentic multi-step retrieval provides high accuracy, citation reliability, and can handle confidential documents",
    "risks": [
        "High implementation complexity",
        "Potential for longer MVP delivery timeline"
    ],
    "mitigations": [
        "Phased implementation approach",
        "Prioritizing key features for MVP"
    ],
    "mvp_plan": [
        "Week 1-2: Design and planning",
        "Week 3-4: Implement query rewriting and reranking",
        "Week 5-6: Implement citation verifi

In [95]:
case_6_2_parsed = extract_json(
    case_6_2_response
)

case_6_2_parsed

{'architecture_scores': [{'option': 'A',
   'score': 8,
   'rationale': 'Balances accuracy and complexity'},
  {'option': 'B',
   'score': 6,
   'rationale': 'Fine-tuning may not be feasible with frequent document changes'},
  {'option': 'C', 'score': 4, 'rationale': 'Lacks accuracy due to no LLM'},
  {'option': 'D',
   'score': 9,
   'rationale': 'High accuracy and citation reliability'}],
 'recommended_architecture': 'Option D',
 'implementation_rationale': 'Agentic multi-step retrieval provides high accuracy, citation reliability, and can handle confidential documents',
 'risks': ['High implementation complexity',
  'Potential for longer MVP delivery timeline'],
 'mitigations': ['Phased implementation approach',
  'Prioritizing key features for MVP'],
 'mvp_plan': ['Week 1-2: Design and planning',
  'Week 3-4: Implement query rewriting and reranking',
  'Week 5-6: Implement citation verification and finalize MVP']}

In [96]:
validate_output(case_6_2_parsed)

True

In [97]:
case_6_2_result = {
    "case_name": "case_6_2_tree_of_thought_architecture",
    "prompt": case_6_2_prompt,
    "raw_response": case_6_2_response,
    "parsed_output": case_6_2_parsed,
    "valid_json": validate_output(case_6_2_parsed)
}

save_output(
    "outputs/tree_of_thought_outputs.json",
    case_6_2_result
)

### Analysis — Case 6.2

#### _Why This Prompt Worked_
- The prompt explicitly encourages branch-wise architectural evaluation before final synthesis.
- Multiple evaluation dimensions prevent the model from blindly selecting the most advanced architecture.
- Instructions penalize unnecessary complexity and unrealistic MVP timelines, reinforcing practical engineering trade-offs.
- The prompt encourages phased implementation thinking, improving realism for startup constraints.
- The schema structure supports organized comparison across architecture strategies.

#### _Potential Failure Modes_
- The model may still overvalue sophisticated agentic architectures despite MVP constraints.
- Fine-tuning approaches may occasionally be recommended despite frequently changing documents.
- Privacy and citation reliability considerations may receive inconsistent weighting.
- Scalability concerns may overshadow short-term delivery feasibility.

#### _Possible Improvements_
- Add explicit scoring dimensions for operational maintenance complexity.
- Introduce weighted scoring for MVP feasibility versus long-term scalability.
- Add infrastructure budget estimates for each architecture option.
- Add explicit latency and operational reliability constraints.

# **`Rephrase-and-Respond`**

## **Case 7.1 — Ambiguous Business Request Rewriting**

In [98]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [99]:
from prompts import case_7_1_prompt

In [100]:
case_7_1_response = call_llm(
    prompt=case_7_1_prompt,
    temperature=0.3
)

In [101]:
print(case_7_1_response)

{
    "rephrased_problem": "Improve operational efficiency by automating manual tasks and enhancing leadership's visibility into team performance metrics",
    "clarified_assumptions": [
        "Manual work refers to repetitive, time-consuming tasks that can be automated",
        "Productivity means reducing task completion time and increasing output quality",
        "Leadership visibility refers to real-time access to key performance indicators (KPIs) and team metrics"
    ],
    "proposed_solution": "Develop a task automation and performance monitoring platform using machine learning and data analytics",
    "target_users": [
        "Team leaders",
        "Operations managers",
        "Frontline staff"
    ],
    "key_features": [
        "Automated task routing and assignment",
        "Predictive analytics for task completion time and quality",
        "Real-time dashboard for leadership to track team performance"
    ],
    "data_needed": [
        "Historical task data",
  

In [102]:
case_7_1_parsed = extract_json(
    case_7_1_response
)

case_7_1_parsed

{'rephrased_problem': "Improve operational efficiency by automating manual tasks and enhancing leadership's visibility into team performance metrics",
 'clarified_assumptions': ['Manual work refers to repetitive, time-consuming tasks that can be automated',
  'Productivity means reducing task completion time and increasing output quality',
  'Leadership visibility refers to real-time access to key performance indicators (KPIs) and team metrics'],
 'proposed_solution': 'Develop a task automation and performance monitoring platform using machine learning and data analytics',
 'target_users': ['Team leaders', 'Operations managers', 'Frontline staff'],
 'key_features': ['Automated task routing and assignment',
  'Predictive analytics for task completion time and quality',
  'Real-time dashboard for leadership to track team performance'],
 'data_needed': ['Historical task data',
  'Team performance metrics',
  'Operational KPIs'],
 'success_metrics': ['Reduction in manual task completion ti

In [103]:
validate_output(case_7_1_parsed)

True

In [104]:
case_7_1_result = {
    "case_name": "case_7_1_rephrase_business_request",
    "prompt": case_7_1_prompt,
    "raw_response": case_7_1_response,
    "parsed_output": case_7_1_parsed,
    "valid_json": validate_output(case_7_1_parsed)
}

In [105]:
case_7_1_result

{'case_name': 'case_7_1_rephrase_business_request',
 'prompt': 'You are an enterprise AI solutions consultant.\n\nYour task is to first clarify an ambiguous business request and then propose a realistic AI solution.\n\nInstructions:\n- First convert vague language into a measurable business problem.\n- Clearly state assumptions used for clarification.\n- Define what productivity and leadership visibility mean in practical operational terms.\n- Propose a focused and realistic AI solution rather than a broad AI transformation program.\n- Avoid generic consulting language.\n- Think step-by-step internally, but return only concise reasoning summaries.\n- Return ONLY valid JSON.\n- Do not include markdown formatting.\n- Do not include explanations outside JSON.\n\nRequired JSON Schema:\n{\n    "rephrased_problem": "",\n    "clarified_assumptions": [],\n    "proposed_solution": "",\n    "target_users": [],\n    "key_features": [],\n    "data_needed": [],\n    "success_metrics": [],\n    "imp

In [106]:
save_output(
    "outputs/rephrase_respond_outputs.json",
    case_7_1_result
)

### Analysis — Case 7.1

#### _Why This Prompt Worked_
- The prompt explicitly separates clarification from solution generation, reinforcing the rephrase-and-respond technique.
- Instructions encourage measurable operational interpretation instead of vague business language.
- The prompt discourages generic AI transformation recommendations and promotes focused solution design.
- The schema structure forces structured decomposition of business goals into users, features, metrics, and risks.
- The reasoning-oriented instructions improve practical business interpretation.

#### _Potential Failure Modes_
- The model may still generate overly broad enterprise AI recommendations.
- Productivity and visibility definitions may remain partially ambiguous.
- Success metrics may become generic if not operationally grounded.
- Some assumptions may be inferred too aggressively without sufficient evidence.

#### _Possible Improvements_
- Add stricter KPI definitions for productivity and leadership visibility.
- Introduce constraints on implementation budget or timeline.
- Add industry-specific operational context for more realistic recommendations.
- Add prioritization ranking for proposed features and implementation phases.

## **Case 7.2 — Rephrase Ambiguous Technical Requirement into Engineering Specification**

In [4]:
import importlib
import prompts

importlib.reload(prompts)

<module 'prompts' from 'c:\\Training\\26-05-2026\\prompt_engineering_evaluation\\prompts.py'>

In [5]:
from prompts import case_7_2_prompt

In [6]:
case_7_2_response = call_llm(
    prompt=case_7_2_prompt,
    temperature=0.3
)

In [7]:
print(case_7_2_response)

{
    "rephrased_requirement": "Develop a scalable AI-powered search system for company documents, enabling employees to efficiently locate information.",
    "functional_requirements": [
        {
            "requirement": "Index company documents for search",
            "description": "Create a system to crawl and index company documents for search functionality"
        },
        {
            "requirement": "Implement search functionality",
            "description": "Develop a search algorithm to retrieve relevant documents based on user queries"
        },
        {
            "requirement": "Support multiple search query types",
            "description": "Allow users to search using various query types, such as keywords, phrases, and natural language"
        },
        {
            "requirement": "Integrate with existing document management system",
            "description": "Integrate the search system with the company's existing document management system for seamless 

In [8]:
case_7_2_parsed = extract_json(
    case_7_2_response
)

case_7_2_parsed

{'rephrased_requirement': 'Develop a scalable AI-powered search system for company documents, enabling employees to efficiently locate information.',
 'functional_requirements': [{'requirement': 'Index company documents for search',
   'description': 'Create a system to crawl and index company documents for search functionality'},
  {'requirement': 'Implement search functionality',
   'description': 'Develop a search algorithm to retrieve relevant documents based on user queries'},
  {'requirement': 'Support multiple search query types',
   'description': 'Allow users to search using various query types, such as keywords, phrases, and natural language'},
  {'requirement': 'Integrate with existing document management system',
   'description': "Integrate the search system with the company's existing document management system for seamless access to documents"}],
 'non_functional_requirements': [{'requirement': 'Scalability',
   'description': 'Ensure the system can handle an increasing 

In [9]:
validate_output(case_7_2_parsed)

True

In [10]:
case_7_2_result = {
    "case_name": "case_7_2_rephrase_engineering_spec",
    "prompt": case_7_2_prompt,
    "raw_response": case_7_2_response,
    "parsed_output": case_7_2_parsed,
    "valid_json": validate_output(case_7_2_parsed)
}

In [11]:
case_7_2_result

{'case_name': 'case_7_2_rephrase_engineering_spec',
 'prompt': 'You are a senior AI systems analyst.\n\nYour task is to convert an ambiguous technical request into a clear engineering specification.\n\nInstructions:\n- First identify ambiguity and missing constraints.\n- Convert vague requirements into measurable engineering objectives.\n- Explicitly separate:\n  - functional requirements\n  - non-functional requirements\n  - assumptions\n  - dependencies\n  - risks\n- Avoid inventing unavailable technical details.\n- Avoid overengineering the solution.\n- Think step-by-step internally, but return only concise reasoning summaries.\n- Return ONLY valid JSON.\n- Do not include markdown formatting.\n- Do not include explanations outside JSON.\n\nRequired JSON Schema:\n{\n    "rephrased_requirement": "",\n    "functional_requirements": [],\n    "non_functional_requirements": [],\n    "assumptions": [],\n    "dependencies": [],\n    "open_questions": [],\n    "risks": [],\n    "recommended_

In [12]:
save_output(
    "outputs/rephrase_respond_outputs.json",
    case_7_2_result
)

### Analysis — Case 7.2

#### _Why This Prompt Worked_
- The prompt explicitly separates ambiguity identification from solution specification.
- Functional and non-functional requirements are isolated into separate structures, improving engineering clarity.
- The prompt discourages hallucinated infrastructure details and unnecessary architectural complexity.
- Open questions and assumptions encourage realistic requirement analysis instead of premature implementation design.
- The schema structure improves traceability between business intent and engineering interpretation.

#### _Potential Failure Modes_
- The model may still over-assume scalability requirements without explicit company size information.
- Some non-functional requirements may remain too generic or difficult to measure.
- Open questions may be incomplete for real enterprise deployments.
- The model may occasionally introduce implicit architecture assumptions.

#### _Possible Improvements_
- Add explicit scaling targets and latency expectations.
- Introduce security and compliance requirements in the original request.
- Add prioritization levels for requirements.
- Include stakeholder-specific requirements for different employee groups.